Notebook 1: Interazione con il sismogramma e picking della curva di dispersione
===============================================================================

In [1]:
# importazione di librerie e moduli
import sys
import copy
import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(1, '../swa')

from geometry import *
from utils import *
from _stream import SeismicStream
from _combineCurves import CombineCurves


import matplotlib
matplotlib.use('Qt5Agg')

In [2]:
# set dei percorsi
prj_dir = '../data/syn_data'
path2raw = os.path.join(prj_dir,'raw')
path2geom = f'{prj_dir}/geometry_test.csv'
ext = '.sgy' # shot file extension

path2proc = os.path.join(prj_dir,'proc')
path2disp = os.path.join(path2proc,'dc')
path2cmb = os.path.join(path2proc,'cmb')

# Check directories
if not os.path.exists(path2proc):
    os.makedirs(path2proc)
    print(f"📁 Directory created: {path2proc}")
if not os.path.exists(path2disp):
    os.makedirs(path2disp)
    print(f"📁 Directory created: {path2disp}")
if not os.path.exists(path2cmb):
    os.makedirs(path2cmb)
    print(f"📁 Directory created: {path2cmb}")

In [3]:
# dizionario con le impostazioni di processing e plot
settings = create_settings_dict(
                         zero_padding=True, freq_step=0.5,  # zero padding
                         normalize = True,local_max = True, # amplitude normalization
                         )


# get paths to shot files, survey geometry from geometry.csv
shot_files, source_coordinates, receiver_coordinates = read_geometry(path2geom)
path2sht = get_shotfiles_from_geometry(path2raw, shot_files, extension = ext, sort_ascending = False)

# stream object
stream = SeismicStream(path2sht[0][0], settings) # record

## 1. Muting
plot del sismogramma e possibilità di effettuare un muting interattivo del segnale indesiderato

In [4]:
# select amd press enter, than e
stream._mute()

You pressed t.
You pressed e. Process stopped.


## 2. Selezione di una finestra
seleziono solo alcuni canali e visualizzo il nuovo sismogramma, utile per le finestrature

In [5]:
# min and max trace
mintrace = 50
maxtrace = 100

trace_select = range(mintrace, maxtrace)
stream._select_traces(trace_select=trace_select)
stream._plotSeismogram(amp_scale=1.)

## 3. Picking DC
picking interattivo dei massimi di ampiezza per estrarre la curva di dispersione. Il massimo è calcolato automaticamente, a noi basta definire alcuni punti con annesso range di selezione.

In [6]:
# processing and plotting settings
trafotype = ['fdbf','phaseshift']

settings = create_settings_dict(trafo = trafotype[0],             # transformation type
                         zero_padding=True, freq_step=0.5, # zero padding
                         normalize = True,local_max = True, # amplitude normalization
                         picking = 'manual',                # picking mode ("manual" or "auto")
                         fmin=1, fmax=50,                  # frequency range
                         vmin=30, vmax=1300, velstep=1)     # testing phase velocity range and step

# get paths to shot files, survey geometry from geometry.csv
shot_files, source_coordinates, receiver_coordinates = read_geometry(path2geom)
path2sht = get_shotfiles_from_geometry(path2raw, shot_files, extension = ext, sort_ascending = False)

In [7]:
# define subset range
mintrace = 1
maxtrace = 24

stream = SeismicStream(path2sht[0][0], settings) # record

# select a subset of the data
trace_select = range(mintrace, maxtrace)
stream_sub = copy.deepcopy(stream)
stream_sub._select_traces(trace_select=trace_select)

# apply the transformation based on the settings in the settings dictionary and do the dispersion curve picking
stream_sub._apply_trafo(do_pick = True,       # start the picking
                        save_dc = True,       # save the resulting pick file
                        path2disp = path2disp)# location to save the dispersion curve

/home/alberto/anaconda3/envs/swa/lib/python3.9/site-packages/matplotlib/contour.py:1568: ComplexWarning: Casting complex values to real discards the imaginary part
  self.zmax = z.max().astype(float)
/home/alberto/anaconda3/envs/swa/lib/python3.9/site-packages/matplotlib/contour.py:1569: ComplexWarning: Casting complex values to real discards the imaginary part
  self.zmin = z.min().astype(float)
/home/alberto/anaconda3/envs/swa/lib/python3.9/site-packages/numpy/ma/core.py:2820: ComplexWarning: Casting complex values to real discards the imaginary part
  _data = np.array(data, dtype=dtype, copy=copy,


Extracting F0 DC curve.
You pressed e. Process stopped.


## 4. Combinazione di più DCs
eseguiamo un loop su tutti i file per estrarre automaticamente le curve di dispersione relative alla stessa posizione xmid

In [10]:
# get paths to shot files, survey geometry from geometry.csv
shot_files, source_coordinates, receiver_coordinates = read_geometry(path2geom)
path2sht = get_shotfiles_from_geometry(path2raw, shot_files, extension = ext, sort_ascending = False)

# define subset range
mintrace = 2
maxtrace = 40

In [11]:
# loop over all files and extract the dispersion curves at the same xmid location
for i in range(len(path2sht)):
    stream = SeismicStream(path2sht[i][0], settings) # record

    # select a subset of the data
    trace_select = range(mintrace, maxtrace)
    stream_sub = copy.deepcopy(stream)
    stream_sub._select_traces(trace_select=trace_select)

    # obtain the dispersion curve
    stream_sub._apply_trafo(save_dc = True,                # save the resulting pick file
                            do_pick = True,                # pick a dc in dispersion image
                            path2disp = path2disp,)        # path to dc

/home/alberto/anaconda3/envs/swa/lib/python3.9/site-packages/matplotlib/contour.py:1568: ComplexWarning: Casting complex values to real discards the imaginary part
  self.zmax = z.max().astype(float)
/home/alberto/anaconda3/envs/swa/lib/python3.9/site-packages/matplotlib/contour.py:1569: ComplexWarning: Casting complex values to real discards the imaginary part
  self.zmin = z.min().astype(float)
/home/alberto/anaconda3/envs/swa/lib/python3.9/site-packages/numpy/ma/core.py:2820: ComplexWarning: Casting complex values to real discards the imaginary part
  _data = np.array(data, dtype=dtype, copy=copy,


Extracting F0 DC curve.
You pressed e. Process stopped.
Extracting F0 DC curve.
You pressed e. Process stopped.
Extracting F0 DC curve.
You pressed e. Process stopped.
Extracting F0 DC curve.
You pressed e. Process stopped.


Ora possiamo effettuare un trimming interattivo per gli outliers e quindi calcoliamo la DC media e deviazione standard

In [12]:
# %% combine dispersion curves
combCurves = CombineCurves(prjdir=path2disp,        # location where dcs are stored
                           path2cmb = path2cmb)     # location where combined dcs shall be stored

# import the data located in path2disp
combCurves._import_data()

# interactive filtering of data
combCurves._filter_all()

# combine all dispersion curves with same xmid
combCurves._combine_all(a=8,            # parameter controlling the wavelength interval
                        xlim = [2,60],  # xlimit for plotting
                        ylim = [50,800],# ylimit for plotting
                        save=True,     # save the combined dc
                        show=True)      # show the combined dc

No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
No artists with labels found to put in legend.  Note that 